# 01 — Data Understanding

**Dataset:** IBM HR Analytics Employee Attrition & Performance
**Goal:** Inspect the raw data before any cleaning or modeling decisions are made.

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
from src import preprocessing, config

pd.set_option('display.max_columns', 50)
df = preprocessing.load_raw_data()
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Y,Yes,11,3,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,Y,No,23,4,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Y,Yes,15,3,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Y,Yes,11,3,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,Y,No,12,3,4,80,1,6,3,3,2,2,2,2


## Shape, dtypes, and target distribution

In [2]:
summary = preprocessing.inspect_data(df)
print('Shape:', summary['shape'])
print('\nTarget distribution:', summary['target_distribution'])
print('\nDuplicate rows:', summary['duplicates'])
print('\nConstant columns:', summary['constant_columns'])
pd.Series(summary['missing_values']).sum(), 'total missing values (should be 0)'

Shape: (1470, 35)

Target distribution: {'No': 1233, 'Yes': 237}

Duplicate rows: 0

Constant columns: ['EmployeeCount', 'Over18', 'StandardHours']


(np.int64(0), 'total missing values (should be 0)')

In [3]:
df.dtypes.value_counts()

int64    26
str       9
Name: count, dtype: int64

**Observations:**
- 1,470 rows, 35 columns, zero missing values, zero duplicate rows.
- Target `Attrition` is imbalanced: ~16% Yes vs ~84% No — this is handled later via
  `class_weight='balanced'` rather than blind oversampling (see notebook 04).
- `EmployeeCount`, `StandardHours`, and `Over18` are constant across all rows and
  carry no signal. `EmployeeNumber` is a row identifier. All four are dropped in
  `clean_data()` to avoid noise / accidental leakage of an ID as a feature.

## Outlier inspection (numerical features)

In [4]:
df[config.NUMERICAL_FEATURES].describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']]

,mean,std,min,25%,50%,75%,max
Age,36.923810,9.135373,18.0,30.0,36.0,43.00,60.0
DailyRate,802.485714,403.509100,102.0,465.0,802.0,1157.00,1499.0
DistanceFromHome,9.192517,8.106864,1.0,2.0,7.0,14.00,29.0
Education,2.912925,1.024165,1.0,2.0,3.0,4.00,5.0
EnvironmentSatisfaction,2.721769,1.093082,1.0,2.0,3.0,4.00,4.0
HourlyRate,65.891156,20.329428,30.0,48.0,66.0,83.75,100.0
JobInvolvement,2.729932,0.711561,1.0,2.0,3.0,3.00,4.0
JobLevel,2.063946,1.106940,1.0,1.0,2.0,3.00,5.0
JobSatisfaction,2.728571,1.102846,1.0,2.0,3.0,4.00,4.0
MonthlyIncome,6502.931293,4707.956783,1009.0,2911.0,4919.0,8379.00,19999.0


**Outlier decision:** Features like `MonthlyIncome`, `YearsAtCompany`, and
`TotalWorkingYears` are right-skewed with legitimate high values (senior, long-tenured
employees) rather than data-entry errors. These are **retained, not removed** —
tree-based models (Random Forest, XGBoost) are robust to skew, and Logistic
Regression uses `StandardScaler` which handles moderate skew adequately for a
comparison baseline. No outlier removal is applied at this stage.

In [5]:
clean = preprocessing.clean_data(df)
print('Cleaned shape:', clean.shape)
clean.head()

Cleaned shape: (1470, 31)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,1,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Yes,11,3,1,0,8,0,1,6,4,0,5
1,49,0,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,No,23,4,4,1,10,3,3,10,7,1,7
2,37,1,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Yes,15,3,2,0,7,3,3,0,0,0,0
3,33,0,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Yes,11,3,3,0,8,3,3,8,7,3,0
4,27,0,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,No,12,3,4,1,6,3,3,2,2,2,2
